In [18]:
%pip install rdflib

from rdflib import Graph, URIRef, Namespace, RDF, RDFS





[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Ex 2

In [19]:
def convert_rdf_formats():
    """Convertit starwars4.nt vers .ttl, .rdf et .json"""
    g = Graph()
    g.parse("starwars4.nt", format="nt")
    print(f"Chargé {len(g)} triplets — conversion en cours...")

    g.serialize(destination="starwars4.ttl", format="turtle")
    g.serialize(destination="starwars4.rdf", format="xml")
    g.serialize(destination="starwars4.json", format="json-ld")

    print("Fichiers créés : starwars4.ttl, starwars4.rdf, starwars4.json")
    print("Observations : .nt = simple, .ttl = compact, .rdf = XML, .json = JSON-LD")

Ex 3

In [20]:
def work_with_prefixes():
    g = Graph()
    ex = Namespace("http://sample/")
    prop = Namespace("http://sample/prop/")
    g.bind("ex", ex); g.bind("prop", prop)

    g.parse("starwars4.ttl", format="turtle")
    print(f"Chargé {len(g)} triplets")

    output = g.serialize(format="turtle")
    with open("starwars4_with_prefixes.ttl", "w") as f:
        f.write(output)

    print("✓ starwars4_with_prefixes.ttl créé")
    print("Q/R :")
    print("- 'ex' → 'foo' : préfixes deviennent 'foo:...'")
    print("- supprimer 'ex' : URIs utilisent le préfixe par défaut ':'")
    print("Aperçu :", output[:300].replace("\n"," ") + "...")

Ex 4

In [21]:
def combine_and_infer():
    g = Graph()
    ex = Namespace("http://sample/")
    prop = Namespace("http://sample/prop/")
    type_ns = Namespace("http://sample/type/")
    g.bind("ex", ex); g.bind("prop", prop); g.bind("type", type_ns)
    g.bind("rdf", RDF); g.bind("rdfs", RDFS)

    g.parse("schema.ttl", format="turtle")
    schema_triples = len(g)
    g.parse("starwars4.ttl", format="turtle")
    total_triples = len(g)
    print(f"Chargé : {schema_triples} (schéma) + {total_triples-schema_triples} (données) = {total_triples} triplets")

    type_statements = [(s,o) for s,p,o in g.triples((None, RDF.type, None))]
    print(f"{len(type_statements)} déclarations rdf:type trouvées")
    print("Remarque : avec un moteur RDFS on inférerait des types via domain/range et subclass.")
    g.serialize(destination="combined_graph.ttl", format="turtle")
    print("✓ combined_graph.ttl créé")

Ex 5

In [22]:
def manipulate_rdf():
    g = Graph()
    ex = Namespace("http://sample/")
    prop = Namespace("http://sample/prop/")
    type_ns = Namespace("http://sample/type/")
    g.bind("ex", ex); g.bind("prop", prop); g.bind("type", type_ns)
    g.bind("rdf", RDF); g.bind("rdfs", RDFS)

    g.parse("schema.ttl", format="turtle")
    g.parse("starwars4.ttl", format="turtle")
    initial_count = len(g)
    print(f"Chargé {initial_count} triplets")

    han = URIRef(ex.HanSolo); leia = URIRef(ex.Leia)
    loves_property = URIRef(prop.loves)
    g.add((han, loves_property, leia))

    g.add((loves_property, RDF.type, RDF.Property))
    creature = URIRef(type_ns.Creature)
    g.add((loves_property, RDFS.domain, creature))
    g.add((loves_property, RDFS.range, creature))
    has_friend = URIRef(prop['has-friend'])
    g.add((loves_property, RDFS.subPropertyOf, has_friend))

    new_count = len(g)
    print(f"Ajoutés : {new_count - initial_count} triplets — total : {new_count}")

    rdf_xml = g.serialize(format="xml")
    with open("starwars_updated.rdf", "wb") as f:
        f.write(rdf_xml.encode('utf-8'))
    g.serialize(destination="starwars_updated.ttl", format="turtle")
    print("Fichiers créés : starwars_updated.rdf, starwars_updated.ttl")

    for s, p, o in g.triples((None, loves_property, None)):
        print(f"  {str(s).split('/')[-1]} aime {str(o).split('/')[-1]}")

Ex 6

In [23]:
def apply_inference_rules():
    g = Graph()
    ex = Namespace("http://sample/")
    prop = Namespace("http://sample/prop/")
    type_ns = Namespace("http://sample/type/")
    g.bind("ex", ex); g.bind("prop", prop); g.bind("type", type_ns)

    g.parse("schema.ttl", format="turtle")
    g.parse("starwars4.ttl", format="turtle")
    initial_count = len(g)
    print(f"Chargé {initial_count} triplets")

    has_friend = URIRef(prop['has-friend'])
    has_enemy = URIRef(prop['has-enemy'])
    has_sister = URIRef(prop['has-sister'])
    has_brother = URIRef(prop['has-brother'])
    has_son = URIRef(prop['has-son'])
    has_father = URIRef(prop['has-father'])

    inferred_triples = []

    # R1 symétrie
    for s, p, o in list(g.triples((None, has_friend, None))):
        if (o, has_friend, s) not in g:
            g.add((o, has_friend, s))
            inferred_triples.append((o, has_friend, s, "symétrie"))

    # R2 transitivité
    friends = list(g.triples((None, has_friend, None)))
    for s1, _, o1 in friends:
        for s2, _, o2 in friends:
            if o1 == s2 and (s1, has_friend, o2) not in g and s1 != o2:
                g.add((s1, has_friend, o2))
                inferred_triples.append((s1, has_friend, o2, "transitivité"))

    # R3 inverse frères/soeurs
    for s, p, o in list(g.triples((None, has_sister, None))):
        if (o, has_brother, s) not in g:
            g.add((o, has_brother, s))
            inferred_triples.append((o, has_brother, s, "inverse"))
    for s, p, o in list(g.triples((None, has_brother, None))):
        if (o, has_sister, s) not in g:
            g.add((o, has_sister, s))
            inferred_triples.append((o, has_sister, s, "inverse"))

    # R4 ennemi d'un ami
    for s1, _, friend in list(g.triples((None, has_friend, None))):
        for s2, _, enemy in list(g.triples((None, has_enemy, None))):
            if friend == s2 and (s1, has_enemy, enemy) not in g:
                g.add((s1, has_enemy, enemy))
                inferred_triples.append((s1, has_enemy, enemy, "ennemi-de-ami"))

    # R5 père de sibling
    for father, _, son in list(g.triples((None, has_son, None))):
        if (son, has_father, father) not in g:
            g.add((son, has_father, father))
    siblings = list(g.triples((None, has_sister, None))) + list(g.triples((None, has_brother, None)))
    for person, _, sibling in siblings:
        for father, _, child in list(g.triples((None, has_son, None))):
            if child == sibling and (father, has_son, person) not in g:
                g.add((father, has_son, person))
                g.add((person, has_father, father))
                inferred_triples.append((father, has_son, person, "sibling-father"))
                inferred_triples.append((person, has_father, father, "sibling-father"))

    final_count = len(g)
    total_inferred = final_count - initial_count
    print(f"Inférés : {total_inferred} nouveaux triplets")
    for idx, (s,p,o,rule) in enumerate(inferred_triples[:5], start=1):
        print(f"{idx}. {str(s).split('/')[-1]} {str(p).split('/')[-1]} {str(o).split('/')[-1]} [{rule}]")

    print("Remarque : la transitivité sur 'has-friend' peut produire des inférences non souhaitées (ami d'un ami ≠ automatiquement ami).")

    g.serialize(destination="starwars_inferred.ttl", format="turtle")
    print("✓ starwars_inferred.ttl créé")
    with open("inference_summary.txt", "w") as f:
        f.write("RÉSUMÉ D'INFERENCE\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Triplets initiaux : {initial_count}\n")
        f.write(f"Triplets finaux : {final_count}\n")
        f.write(f"Triplets inférés : {total_inferred}\n\n")
        f.write("Exemples :\n")
        for s, p, o, rule in inferred_triples[:50]:
            f.write(f"{str(s).split('/')[-1]} {str(p).split('/')[-1]} {str(o).split('/')[-1]} [{rule}]\n")
    print("✓ inference_summary.txt créé")

In [32]:
def main(run_list=None):
    """
    run_list: liste optionnelle parmi
      'convert','prefixes','combine','manipulate','inference'
    Si None -> exécute tout.
    """
    mapping = {
        "convert": convert_rdf_formats,
        "prefixes": work_with_prefixes,
        "combine": combine_and_infer,
        "manipulate": manipulate_rdf,
        "inference": apply_inference_rules,
    }
    if run_list is None:
        run_list = list(mapping.keys())

    print("=== Lancement des exercices ===")
    for name in run_list:
        print(f"> Exécution : {name}" )
        mapping[name]()
        
        print("\n" )
    print("=== Terminé ===")

In [33]:
main()

=== Lancement des exercices ===
> Exécution : convert
Chargé 14 triplets — conversion en cours...
Fichiers créés : starwars4.ttl, starwars4.rdf, starwars4.json
Observations : .nt = simple, .ttl = compact, .rdf = XML, .json = JSON-LD


> Exécution : prefixes
Chargé 14 triplets
✓ starwars4_with_prefixes.ttl créé
Q/R :
- 'ex' → 'foo' : préfixes deviennent 'foo:...'
- supprimer 'ex' : URIs utilisent le préfixe par défaut ':'
Aperçu : @prefix ex: <http://sample/> . @prefix ns1: <http://sample/prop/> .  ex:StarWarsIV ns1:has-character ex:DarthVader,         ex:HanSolo,         ex:Leia,         ex:LukeSkywalker .  ex:MarkHamill ns1:age "67" .  ex:DarthVader ns1:has-son ex:LukeSkywalker ;     ns1:played-by ex:DavidProwse .  ex:Leia ...


> Exécution : combine
Chargé : 26 (schéma) + 14 (données) = 40 triplets
16 déclarations rdf:type trouvées
Remarque : avec un moteur RDFS on inférerait des types via domain/range et subclass.
✓ combined_graph.ttl créé


> Exécution : manipulate
Chargé 40 triple